# Certificación Profesional en Desarrollador de Agentes de IA
## Clase 8 — Evaluación y mejora de RAG

ITBA Educación Ejecutiva

Medir antes de optimizar: hasta ahora diagnosticamos caso por caso (Clase 7) — hoy construimos un dataset de evaluación y métricas, para saber objetivamente si una mejora realmente mejora algo.

### Cómo se usa este notebook

- El docente corre y explica cada celda en vivo. No programás en simultáneo durante la clase.
- Celdas **🏠 Practicá**: tarea individual para después de la clase.
- **Sección 11 (Laboratorio)**: checkpoint del proyecto integrador.

## Objetivo de la clase

Al final de la clase vas a poder:
- Construir un dataset de evaluación con evidencia esperada por pregunta.
- Calcular métricas de recuperación (hit rate, recall, precision@k, MRR) y de generación (fidelidad, relevancia) con LLM-as-a-Judge.
- Medir costo y latencia agregados, y clasificar errores por etapa.
- Aplicar búsqueda híbrida (BM25 + semántica) y reranking, y demostrar con números si mejoran el sistema.

**Checkpoint del proyecto integrador de hoy**: dataset de evaluación armado y métricas base (fidelidad, relevancia, costo, latencia) medidas (ver `proyecto-integrador/README.md`, checkpoint Clase 8).

## Agenda

1. Medir antes de optimizar y construcción del dataset de evaluación
2. Métricas de recuperación: hit rate, recall, precision@k, MRR
3. Métricas de generación: fidelidad y relevancia (LLM-as-a-Judge)
4. Evaluación humana vs. automática
5. Costo y latencia agregados
6. Análisis de errores por etapa
7. RAGAS (contenido complementario)
8. Mejora: búsqueda híbrida (BM25 + semántica)
9. Mejora: reranking
10. Aplicar una mejora e informe comparativo
11. Laboratorio: medición del sistema base y comparación

## 0. Punto de partida

Reconstruimos el pipeline RAG de las Clases 6-7 (mismo corpus de juguete, con el chunk redundante `apis-003`) para tener algo completo sobre qué medir.

In [ ]:
import os, json, time
from dotenv import load_dotenv
load_dotenv()

import boto3
bedrock = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))
MODEL_ID = os.environ.get("BEDROCK_MODEL_ID_CLAUDE", "global.anthropic.claude-sonnet-4-6")
EMBED_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID_EMBED", "cohere.embed-v4:0")

def embed_bedrock(textos, input_type="search_document"):
    body = json.dumps({"texts": textos, "input_type": input_type, "embedding_types": ["float"]})
    resp = bedrock.invoke_model(modelId=EMBED_MODEL_ID, body=body, accept="*/*", contentType="application/json")
    return json.loads(resp["body"].read())["embeddings"]["float"]

def llamar_bedrock(prompt, system=None, temperature=0.0, max_tokens=500):
    kwargs = {"modelId": MODEL_ID, "messages": [{"role": "user", "content": [{"text": prompt}]}],
              "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens}}
    if system:
        kwargs["system"] = [{"text": system}]
    resp = bedrock.converse(**kwargs)
    return resp["output"]["message"]["content"][0]["text"], resp["usage"], resp["metrics"]["latencyMs"]

In [ ]:
import chromadb

corpus = [
    {"id": "apis-001", "texto": "La Converse API de Bedrock unifica la forma de llamar a distintos modelos: cambia el modelId, no el código.", "doc": "guia-apis"},
    {"id": "apis-002", "texto": "Los parámetros de inferencia como temperature y maxTokens van en inferenceConfig en Bedrock.", "doc": "guia-apis"},
    {"id": "apis-003", "texto": "La temperatura y el máximo de tokens se configuran dentro del campo inferenceConfig de Bedrock.", "doc": "guia-apis"},
    {"id": "seguridad-001", "texto": "La inyección de prompts ocurre cuando contenido no confiable manipula las instrucciones del modelo.", "doc": "guia-seguridad"},
    {"id": "seguridad-002", "texto": "El principio de mínimo privilegio limita qué herramientas puede invocar un agente según el contexto.", "doc": "guia-seguridad"},
    {"id": "rag-001", "texto": "Un pipeline RAG combina recuperación de evidencia con generación sustentada en esa evidencia.", "doc": "guia-rag"},
    {"id": "rag-002", "texto": "El top-k determina cuántos fragmentos recuperados se incluyen como contexto para el modelo generador.", "doc": "guia-rag"},
]

cliente_chroma = chromadb.Client()
coleccion = cliente_chroma.get_or_create_collection("corpus_clase8")
coleccion.add(
    ids=[c["id"] for c in corpus],
    embeddings=embed_bedrock([c["texto"] for c in corpus]),
    documents=[c["texto"] for c in corpus],
    metadatas=[{"doc": c["doc"]} for c in corpus],
)

def recuperar(consulta, k=3):
    vec = embed_bedrock([consulta], input_type="search_query")[0]
    r = coleccion.query(query_embeddings=[vec], n_results=k)
    return [{"chunk_id": i, "texto": t, "doc": m["doc"], "distancia": d}
            for i, t, m, d in zip(r["ids"][0], r["documents"][0], r["metadatas"][0], r["distances"][0])]

def rag(consulta, k=3, umbral=1.0):
    chunks = [c for c in recuperar(consulta, k) if c["distancia"] < umbral]
    if not chunks:
        return {"respuesta": "No encontré evidencia suficiente para responder.", "chunks": [], "usage": None, "latencia_ms": 0}
    contexto = "\n\n".join(f"[{i+1}] {c['texto']}" for i, c in enumerate(chunks))
    prompt = f"<evidencia>\n{contexto}\n</evidencia>\n\n<pregunta>\n{consulta}\n</pregunta>\n\nRespondé usando solo la evidencia."
    system = "Respondé EXCLUSIVAMENTE con la evidencia provista. Si no alcanza, decilo explícitamente."
    respuesta, usage, latencia = llamar_bedrock(prompt, system=system)
    return {"respuesta": respuesta, "chunks": chunks, "usage": usage, "latencia_ms": latencia}

print("Pipeline reconstruido. Corpus:", coleccion.count(), "chunks")

## 1. Medir antes de optimizar y dataset de evaluación

Sin un dataset de evaluación, "mejorar" el RAG es adivinar — cambiás algo, te parece que anduvo mejor en 2-3 pruebas sueltas, y seguís. Un dataset fijo (preguntas + evidencia esperada) permite comparar versiones de forma objetiva y repetible.

Cada caso necesita, como mínimo: la pregunta, qué chunk(s) deberían recuperarse, y si se espera que el sistema responda o diga "no sé".

In [ ]:
dataset_evaluacion = [
    {"pregunta": "¿Qué parámetro controla la temperatura de generación?", "chunks_relevantes": ["apis-002", "apis-003"], "debe_responder": True},
    {"pregunta": "¿Qué es la inyección de prompts?", "chunks_relevantes": ["seguridad-001"], "debe_responder": True},
    {"pregunta": "¿Qué elementos describen un pipeline RAG?", "chunks_relevantes": ["rag-001", "rag-002"], "debe_responder": True},
    {"pregunta": "¿Qué es el principio de mínimo privilegio?", "chunks_relevantes": ["seguridad-002"], "debe_responder": True},
    {"pregunta": "¿Cómo se hace un backup de la base de datos?", "chunks_relevantes": [], "debe_responder": False},
]
print(f"{len(dataset_evaluacion)} casos en el dataset de evaluación.")

**🏠 Practicá**: agregá 3 casos más sobre tu corpus real del TP (MDN Web Docs), incluyendo uno con más de un chunk relevante — lo vas a necesitar para el laboratorio.

## 2. Métricas de recuperación

- **Hit rate@k**: ¿al menos un chunk relevante está en el top-k? (sí/no por pregunta, promediado)
- **Recall@k**: de los chunks relevantes que existen, ¿qué fracción se recuperó?
- **Precision@k**: de los k chunks recuperados, ¿qué fracción es relevante?
- **MRR** (Mean Reciprocal Rank): en qué posición apareció el primer chunk relevante (1/posición, promediado) — premia que lo relevante esté *arriba*, no solo presente.

In [ ]:
def metricas_recuperacion(dataset, k=3):
    hits, recalls, precisions, reciprocal_ranks = [], [], [], []

    for caso in dataset:
        if not caso["chunks_relevantes"]:
            continue  # estos casos se evalúan aparte (sección 4: ¿dijo "no sé" correctamente?)

        recuperados = [c["chunk_id"] for c in recuperar(caso["pregunta"], k=k)]
        relevantes = set(caso["chunks_relevantes"])
        recuperados_relevantes = [r for r in recuperados if r in relevantes]

        hits.append(1 if recuperados_relevantes else 0)
        recalls.append(len(set(recuperados_relevantes)) / len(relevantes))
        precisions.append(len(recuperados_relevantes) / k)

        rr = 0.0
        for pos, r in enumerate(recuperados, start=1):
            if r in relevantes:
                rr = 1 / pos
                break
        reciprocal_ranks.append(rr)

    return {
        "hit_rate": sum(hits) / len(hits),
        "recall": sum(recalls) / len(recalls),
        "precision": sum(precisions) / len(precisions),
        "mrr": sum(reciprocal_ranks) / len(reciprocal_ranks),
    }


metricas_base = metricas_recuperacion(dataset_evaluacion, k=3)
for nombre, valor in metricas_base.items():
    print(f"{nombre}: {valor:.3f}")

Notá la diferencia entre recall y precision en el caso "¿Qué elementos describen un pipeline RAG?" (2 chunks relevantes, `rag-001` y `rag-002`): si el top-3 trae ambos más un chunk irrelevante, recall=100% pero precision=66%. Son medidas de cosas distintas — alto recall con baja precision significa "no te perdiste nada, pero metiste ruido".

**🏠 Practicá**: corré `metricas_recuperacion` con k=1 y k=5, y compará. ¿Qué métrica sube y cuál baja al aumentar k? ¿Tiene sentido con lo que viste en la Clase 7 sobre ajuste de top-k?

## 3. Métricas de generación: fidelidad y relevancia (LLM-as-a-Judge)

Las métricas de recuperación miden si *llegó* la evidencia correcta — no si la *respuesta* la usó bien. Para eso hace falta evaluar el texto generado, y la forma más práctica de automatizarlo es usar otro LLM como juez: **LLM-as-a-Judge**.

- **Fidelidad (faithfulness)**: ¿la respuesta está sustentada en la evidencia, sin agregar nada que no esté ahí?
- **Relevancia (answer relevance)**: ¿la respuesta realmente contesta la pregunta (más allá de estar sustentada)?

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class JuicioLLM(BaseModel):
    fidelidad: Literal[1, 2, 3, 4, 5] = Field(..., description="1=inventa información no presente en la evidencia, 5=totalmente sustentada")
    relevancia: Literal[1, 2, 3, 4, 5] = Field(..., description="1=no contesta la pregunta, 5=la contesta completamente")
    justificacion: str = Field(..., description="Una oración explicando el puntaje")


SYSTEM_JUEZ = (
    "Sos un evaluador estricto de sistemas RAG. Vas a recibir una pregunta, la evidencia que tenía disponible "
    "el sistema, y la respuesta que generó. Evaluá fidelidad y relevancia según la evidencia, no según tu "
    "propio conocimiento. Respondé ÚNICAMENTE con un JSON que cumpla el schema provisto."
)

def juzgar(pregunta: str, evidencia: str, respuesta: str) -> JuicioLLM:
    prompt = (
        f"Pregunta: {pregunta}\n\nEvidencia disponible:\n{evidencia}\n\nRespuesta generada:\n{respuesta}\n\n"
        f"Schema: {JuicioLLM.model_json_schema()}"
    )
    texto, _, _ = llamar_bedrock(prompt, system=SYSTEM_JUEZ, temperature=0.0)
    return JuicioLLM.model_validate_json(texto)


caso = dataset_evaluacion[0]
resultado = rag(caso["pregunta"])
evidencia_texto = "\n".join(c["texto"] for c in resultado["chunks"])
juicio = juzgar(caso["pregunta"], evidencia_texto, resultado["respuesta"])
print(f"Fidelidad: {juicio.fidelidad}/5  Relevancia: {juicio.relevancia}/5")
print(f"Justificación: {juicio.justificacion}")

Notá que `juzgar` reutiliza structured outputs + Pydantic de la Clase 3 — LLM-as-a-Judge no es más que otra llamada al modelo con un schema de salida, aplicada a evaluar en vez de a responder.

**🏠 Practicá**: corré `juzgar` sobre los 4 casos del dataset que esperan respuesta y promediá fidelidad/relevancia — es tu primera métrica de generación agregada.

## 4. Evaluación humana vs. automática

LLM-as-a-Judge es rápido y escala, pero tiene sesgos propios: tiende a calificar más generosamente que un humano, puede ser inconsistente entre corridas (sobre todo con temperature > 0), y hereda cualquier punto ciego del modelo usado como juez.

En la práctica: usar LLM-as-a-Judge para iterar rápido (como hoy), pero validar con una muestra de evaluación humana antes de confiar en una mejora para producción — sobre todo si el juez y el modelo evaluado son el mismo proveedor (hay evidencia de que los modelos tienden a calificarse mejor a sí mismos que a otros).

## 5. Costo y latencia agregados

Una mejora que sube la fidelidad 10% pero duplica el costo por consulta puede no valer la pena — siempre se evalúa junto con costo y latencia, no la calidad sola.

In [ ]:
PRECIO_POR_MILLON_USD = {"input": 3.0, "output": 15.0}  # valores de ejemplo, verificar precio vigente

def costo_usd(usage):
    if not usage:
        return 0.0
    return usage["inputTokens"] / 1_000_000 * PRECIO_POR_MILLON_USD["input"] + usage["outputTokens"] / 1_000_000 * PRECIO_POR_MILLON_USD["output"]


costos, latencias = [], []
for caso in dataset_evaluacion:
    r = rag(caso["pregunta"])
    costos.append(costo_usd(r["usage"]))
    latencias.append(r["latencia_ms"])

print(f"Costo total del dataset: ${sum(costos):.5f}  (promedio: ${sum(costos)/len(costos):.5f} por consulta)")
print(f"Latencia promedio: {sum(latencias)/len(latencias):.0f} ms")

## 6. Análisis de errores por etapa

Reutilizamos el framework de la Clase 7: para cada caso del dataset con métricas bajas, clasificar en qué etapa está el problema antes de intentar arreglar nada.

In [ ]:
def clasificar_error(caso, k=3):
    if not caso["chunks_relevantes"]:
        return None  # casos "no debe responder" se evalúan distinto

    recuperados = [c["chunk_id"] for c in recuperar(caso["pregunta"], k=k)]
    if not any(r in caso["chunks_relevantes"] for r in recuperados):
        return "RECUPERACIÓN: ningún chunk relevante llegó al top-k"

    resultado = rag(caso["pregunta"], k=k)
    chunks_en_contexto = [c["chunk_id"] for c in resultado["chunks"]]
    if not any(r in chunks_en_contexto for r in caso["chunks_relevantes"]):
        return "CONTEXTO: el chunk relevante se recuperó pero no pasó el umbral"

    juicio = juzgar(caso["pregunta"], "\n".join(c["texto"] for c in resultado["chunks"]), resultado["respuesta"])
    if juicio.fidelidad < 4:
        return f"GENERACIÓN: fidelidad baja ({juicio.fidelidad}/5) pese a tener la evidencia correcta"

    return None  # sin error detectado


for caso in dataset_evaluacion:
    diagnostico = clasificar_error(caso)
    if diagnostico:
        print(f"⚠ {caso['pregunta']}\n  -> {diagnostico}")
    else:
        print(f"✓ {caso['pregunta']}")

## 7. RAGAS (contenido complementario)

Lo que armamos a mano en las secciones 2-3 (hit rate, recall, fidelidad, relevancia) son, esencialmente, versiones simplificadas de métricas que **RAGAS** (librería open source especializada en evaluación de RAG) ya implementa de forma estandarizada: `faithfulness`, `answer_relevancy`, `context_precision`, `context_recall`, entre otras.

No la instalamos hoy (trae dependencias pesadas y su propia curva de aprendizaje de configuración), pero vale la pena conocerla para proyectos reales: una vez que entendés qué mide cada métrica a mano (que es el objetivo de hoy), adoptar RAGAS es sobre todo una cuestión de wiring, no de concepto nuevo. Si te interesa, `pip install ragas` y revisá su documentación oficial — la API cambia seguido, así que no vale la pena memorizar código específico acá.

## 8. Mejora: búsqueda híbrida (BM25 + semántica)

La búsqueda semántica (embeddings) es buena para significado, pero puede fallar con términos exactos poco frecuentes (nombres de funciones, códigos de error) que casi no tienen "significado" distintivo en el espacio vectorial. **BM25** es un algoritmo clásico de búsqueda por palabras clave (basado en frecuencia de términos) que es complementario, no competidor.

**Búsqueda híbrida**: combinar el ranking de ambos métodos (por ejemplo, con *Reciprocal Rank Fusion*) en vez de elegir uno solo.

In [ ]:
# %pip install rank_bm25
from rank_bm25 import BM25Okapi

corpus_tokenizado = [c["texto"].lower().split() for c in corpus]
bm25 = BM25Okapi(corpus_tokenizado)

def buscar_bm25(consulta: str, k: int = 3) -> list[dict]:
    scores = bm25.get_scores(consulta.lower().split())
    indices_top = sorted(range(len(scores)), key=lambda i: -scores[i])[:k]
    return [{"chunk_id": corpus[i]["id"], "score_bm25": scores[i]} for i in indices_top]


def busqueda_hibrida(consulta: str, k: int = 3, k_fusion: int = 60) -> list[str]:
    """Reciprocal Rank Fusion: combina el ranking semántico y el de BM25."""
    ranking_semantico = [c["chunk_id"] for c in recuperar(consulta, k=len(corpus))]
    ranking_bm25 = [r["chunk_id"] for r in buscar_bm25(consulta, k=len(corpus))]

    scores_fusion = {}
    for ranking in (ranking_semantico, ranking_bm25):
        for pos, chunk_id in enumerate(ranking, start=1):
            scores_fusion[chunk_id] = scores_fusion.get(chunk_id, 0) + 1 / (k_fusion + pos)

    return sorted(scores_fusion, key=lambda cid: -scores_fusion[cid])[:k]


print("Híbrido:", busqueda_hibrida("¿cómo configuro el inferenceConfig?", k=3))
print("Solo semántico:", [c["chunk_id"] for c in recuperar("¿cómo configuro el inferenceConfig?", k=3)])

**🏠 Practicá**: probá una consulta con un término técnico muy específico y poco frecuente (ej. el nombre exacto de un parámetro, "maxTokens") y compará búsqueda híbrida vs. solo semántica — es el caso donde BM25 suele ganar, porque coincide la palabra exacta.

## 9. Mejora: reranking

Ya vimos el concepto en la Clase 1 (bi-encoder vs. cross-encoder). AWS Bedrock expone un **modelo de reranking real** (Cohere Rerank 3.5) vía una API dedicada — se usa como segunda etapa, sobre los candidatos que ya trajo la recuperación inicial (semántica o híbrida).

In [ ]:
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))

def rerank(consulta: str, candidatos: list[dict], top_n: int = 3) -> list[dict]:
    response = bedrock_agent_runtime.rerank(
        queries=[{"type": "TEXT", "textQuery": {"text": consulta}}],
        sources=[
            {"type": "INLINE", "inlineDocumentSource": {"type": "TEXT", "textDocument": {"text": c["texto"]}}}
            for c in candidatos
        ],
        rerankingConfiguration={
            "type": "BEDROCK_RERANKING_MODEL",
            "bedrockRerankingConfiguration": {
                "modelConfiguration": {"modelArn": "arn:aws:bedrock:us-east-1::foundation-model/cohere.rerank-v3-5:0"},
                "numberOfResults": top_n,
            },
        },
    )
    return [{"chunk_id": candidatos[r["index"]]["chunk_id"], "relevance_score": r["relevanceScore"]} for r in response["results"]]


candidatos = recuperar("¿qué controla la temperatura de generación?", k=6)  # traemos más candidatos de los que necesitamos
reordenados = rerank("¿qué controla la temperatura de generación?", candidatos, top_n=3)
for r in reordenados:
    print(f"{r['chunk_id']}  (score={r['relevance_score']:.3f})")

Notá el patrón: recuperación inicial trae k_grande candidatos "razonablemente relevantes" rápido (bi-encoder, barato), reranking reordena con un modelo más preciso pero más lento (cross-encoder) — exactamente el trade-off de la Clase 1. No conviene rerankear todo el corpus, solo el top de la recuperación inicial.

**🏠 Practicá**: compará el orden de `reordenados` contra el orden semántico simple de `candidatos` — ¿cambió algo? Con nuestro corpus de juguete (pocos candidatos, bien separados) puede no notarse mucho; con el corpus real del TP debería ser más evidente.

## 10. Aplicar una mejora e informe comparativo

El punto de todo lo anterior: no alcanza con "creo que el reranking ayuda" — hay que medirlo contra el mismo dataset de evaluación, antes y después.

In [ ]:
def metricas_con_reranking(dataset, k_candidatos=6, k_final=3):
    hits = []
    for caso in dataset:
        if not caso["chunks_relevantes"]:
            continue
        candidatos = recuperar(caso["pregunta"], k=k_candidatos)
        reordenados = rerank(caso["pregunta"], candidatos, top_n=k_final)
        recuperados = [r["chunk_id"] for r in reordenados]
        hits.append(1 if any(r in caso["chunks_relevantes"] for r in recuperados) else 0)
    return {"hit_rate": sum(hits) / len(hits)}


metricas_mejoradas = metricas_con_reranking(dataset_evaluacion)

print("--- Informe comparativo ---")
print(f"Hit rate ANTES (solo semántico, k=3):     {metricas_base['hit_rate']:.3f}")
print(f"Hit rate DESPUÉS (con reranking, k=3):     {metricas_mejoradas['hit_rate']:.3f}")

Con un corpus de juguete de 7 chunks, es esperable que no haya diferencia (ya partíamos de hit rate alto) — el valor real de este informe aparece con el corpus real del TP, donde hay más margen para que una mejora se note en los números. El punto pedagógico es el **proceso**: medir antes, aplicar, medir después, comparar — no la magnitud del número en este ejemplo chico.

## 11. Laboratorio: medición del sistema base y comparación

**🏠 Tarea de la semana — entregar/completar antes de la Clase 9.**

**Checkpoint del proyecto integrador (Clase 8)**: dataset de evaluación armado y métricas base (fidelidad, relevancia, costo, latencia) medidas sobre tu RAG real.

### Consigna

1. Armá un dataset de evaluación de al menos 10 preguntas sobre tu corpus real (MDN Web Docs), con `chunks_relevantes` esperados — incluí casos con un solo chunk relevante, casos con varios, y al menos 2 fuera de dominio.
2. Corré `metricas_recuperacion` (hit rate, recall, precision, MRR) sobre tu sistema actual (el de la Clase 7).
3. Corré LLM-as-a-Judge (fidelidad y relevancia) sobre los casos que sí deberían responder, y calculá el promedio.
4. Medí costo total y latencia promedio del dataset completo.
5. Elegí **una** mejora (búsqueda híbrida o reranking) y volvé a medir — armá un informe comparativo antes/después con los 4 números (hit rate/MRR, fidelidad/relevancia promedio, costo, latencia).

### Criterio de éxito

- Tenés un número concreto para cada métrica, no una impresión cualitativa ("me pareció que mejoró").
- El informe comparativo muestra explícitamente si la mejora elegida valió la pena considerando también costo/latencia, no solo calidad.
- Podés señalar, para al menos un caso del dataset, en qué etapa está el error (sección 6) si todavía falla después de la mejora.

In [ ]:
def evaluar_sistema(dataset, pipeline_rag) -> dict:
    # TODO: calcular métricas de recuperación (metricas_recuperacion, adaptada a tu pipeline real)
    # TODO: calcular fidelidad/relevancia promedio con LLM-as-a-Judge sobre los casos con respuesta esperada
    # TODO: calcular costo total y latencia promedio
    # TODO: devolver un dict con las 4 categorías de métricas
    raise NotImplementedError


# Prueba (va a fallar hasta que completes el TODO):
# metricas_antes = evaluar_sistema(tu_dataset, tu_rag_sin_mejora)
# metricas_despues = evaluar_sistema(tu_dataset, tu_rag_con_mejora)
# print(metricas_antes, metricas_despues)

## Cierre

- Hoy pasamos de "diagnostiqué caso por caso" (Clase 7) a medir con métricas sobre un dataset completo — recuperación, generación, costo y latencia.
- Toda mejora de acá en adelante (en el TP, y en cualquier sistema RAG real) se evalúa con este mismo patrón: medir antes, aplicar una cosa a la vez, medir después, comparar.
- Próxima clase: herramientas y workflows agénticos — el RAG deja de ser lo único que el sistema sabe hacer.
- Antes de la próxima clase: dejar el laboratorio de la sección 11 funcionando, con el informe comparativo de al menos una mejora.